In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

In [16]:
OUT_PUT = "./data/flight.parquet"
 
df = pd.read_parquet("data/drone.parquet", engine="pyarrow")
df

,flight,time,wind_speed,wind_angle,battery_voltage,battery_current,position_x,position_y,position_z,orientation_x,...,angular_z,linear_acceleration_x,linear_acceleration_y,linear_acceleration_z,speed,payload,altitude,date,time_day,route
0,1,0.00,0.1,12.0,24.222174,0.087470,-79.782396,40.458047,269.332402,0.001772,...,0.006815,0.004258,-0.120405,-9.811137,4.0,0.0,25,2019-04-07,10:13,R5
1,1,0.20,0.1,3.0,24.227180,0.095421,-79.782396,40.458047,269.332056,0.001768,...,0.002034,0.006175,-0.116397,-9.810392,4.0,0.0,25,2019-04-07,10:13,R5
2,1,0.30,0.1,352.0,24.225929,0.095421,-79.782396,40.458047,269.333081,0.001768,...,-0.000874,0.002696,-0.128592,-9.809440,4.0,0.0,25,2019-04-07,10:13,R5
3,1,0.50,0.1,354.0,24.224678,0.095421,-79.782396,40.458047,269.334648,0.001775,...,0.002443,0.002024,-0.128271,-9.810159,4.0,0.0,25,2019-04-07,10:13,R5
4,1,0.60,0.1,359.0,24.210905,0.079518,-79.782396,40.458047,269.336178,0.001775,...,-0.006425,0.008271,-0.119890,-9.812125,4.0,0.0,25,2019-04-07,10:13,R5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257891,279,152.10,1.1,198.0,22.857437,0.095421,-79.782802,40.459018,271.560190,0.021382,...,0.009449,0.444553,-0.274965,-9.796700,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257892,279,152.20,1.1,196.0,22.847422,0.095421,-79.782802,40.459018,271.571983,0.021383,...,-0.001755,0.451230,-0.240619,-9.793810,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257893,279,152.41,1.2,189.0,22.856186,0.111325,-79.782802,40.459018,271.584533,0.021385,...,0.008545,0.443839,-0.274903,-9.796004,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257894,279,152.60,1.1,187.0,22.854933,0.127228,-79.782802,40.459018,271.588050,0.021393,...,-0.001379,0.443880,-0.248434,-9.794703,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 257896 entries, 0 to 257895
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   flight                 257896 non-null  int64   
 1   time                   257896 non-null  float64 
 2   wind_speed             257896 non-null  float64 
 3   wind_angle             257896 non-null  float64 
 4   battery_voltage        257896 non-null  float64 
 5   battery_current        257896 non-null  float64 
 6   position_x             257896 non-null  float64 
 7   position_y             257896 non-null  float64 
 8   position_z             257896 non-null  float64 
 9   orientation_x          257896 non-null  float64 
 10  orientation_y          257896 non-null  float64 
 11  orientation_z          257896 non-null  float64 
 12  orientation_w          257896 non-null  float64 
 13  velocity_x             257896 non-null  float64 
 14  velocity_y             257896 n

In [18]:
temp = df.select_dtypes(include= 'number').agg(['min', 'max', 'mean', 'median', 'std']).T
temp

,min,max,mean,median,std
flight,1.000000e+00,279.000000,162.445513,166.000000,71.445026
time,0.000000e+00,428.290000,97.766940,93.000000,62.667401
wind_speed,0.000000e+00,18.100000,4.337992,3.300000,3.494649
wind_angle,0.000000e+00,359.000000,169.470760,177.000000,101.176006
battery_voltage,1.881081e+01,25.894913,22.435459,22.306536,1.200742
battery_current,-3.260227e-01,47.193779,17.932477,21.923040,10.865654
position_x,-7.994654e+01,0.000000,-77.773754,-79.782747,12.506800
position_y,0.000000e+00,40.459682,39.438805,40.458994,6.342148
position_z,0.000000e+00,376.188721,295.812932,292.120051,58.305928
orientation_x,-5.464130e-01,0.414962,-0.010096,-0.001910,0.047414


### Feature engineering

In [14]:
del df
gc.collect()
%whos

Variable         Type                Data/Info
----------------------------------------------
OUT_PUT          str                 ./data/flight.parquet
bad              list                n=3
flight_summary   DataFrame           Shape: (209, 21)
g                DataFrameGroupBy    <pandas.api.typing.DataFr<...>object at 0x7f9084779c70>
gc               module              <module 'gc' (built-in)>
np               module              <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd               module              <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
plt              module              <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
sns              module              <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>
temp             DataFrame           Shape: (24, 5)
total_capacity   Series              Shape: (209,)


In [19]:
df = df.sort_values(["flight", "time"]).reset_index(drop=True)
g = df.groupby("flight", group_keys=False)

In [20]:
df["dt"] = g["time"].diff().fillna(0)
 
# battery_current is in Amps, dt in seconds -> capacity_step is in mAh
# (A * s) / 3.6 = mAh.  Summing this over a flight gives total mAh consumed.
df["capacity_step"] = 1 / 3.6 * df["dt"] * df["battery_current"]
 
df["wind_x"] = df["wind_speed"] * np.sin(np.deg2rad(df["wind_angle"]))
df["wind_y"] = df["wind_speed"] * np.cos(np.deg2rad(df["wind_angle"]))
 
df["velocity_magnitude"] = np.linalg.norm(
    df[["velocity_x", "velocity_y", "velocity_z"]], axis=1
)
df["acceleration_magnitude"] = np.linalg.norm(
    df[["linear_acceleration_x", "linear_acceleration_y", "linear_acceleration_z"]],
    axis=1,
)
 
# extra helper: per-step 3D distance travelled, needed to build a "range" feature
df["step_dx"] = g["position_x"].diff().fillna(0)
df["step_dy"] = g["position_y"].diff().fillna(0)
df["step_dz"] = g["position_z"].diff().fillna(0)
df["step_dist"] = np.sqrt(df["step_dx"] ** 2 + df["step_dy"] ** 2 + df["step_dz"] ** 2)
 

In [21]:
flight_summary = g.agg(
    payload=("payload", "first"),
    route=("route", "first"),
    altitude=("altitude", "first"),
    date=("date", "first"),
    time_day=("time_day", "first"),
 
    duration_s=("time", "max"),                       # total flight time
    total_distance_m=("step_dist", "sum"),             # proxy for "range"
 
    wind_speed_mean=("wind_speed", "mean"),
    wind_speed_std=("wind_speed", "std"),
    wind_x_mean=("wind_x", "mean"),
    wind_y_mean=("wind_y", "mean"),
 
    speed_mean=("speed", "mean"),
    speed_max=("speed", "max"),                        # acts as the "speed limit" reached
 
    velocity_mag_mean=("velocity_magnitude", "mean"),
    velocity_mag_max=("velocity_magnitude", "max"),
 
    acceleration_mag_mean=("acceleration_magnitude", "mean"),
    acceleration_mag_std=("acceleration_magnitude", "std"),
 
    battery_voltage_mean=("battery_voltage", "mean"),
    battery_voltage_min=("battery_voltage", "min"),
 
    battery_needed=("capacity_step", "sum"),           # <-- TARGET (mAh consumed)
).reset_index()
 
flight_summary["wind_speed_std"] = flight_summary["wind_speed_std"].fillna(0)
flight_summary["acceleration_mag_std"] = flight_summary["acceleration_mag_std"].fillna(0)

In [22]:
bad = flight_summary.loc[flight_summary["battery_needed"] < 0, "flight"].tolist()
if bad:
    print(f"[warning] {len(bad)} flight(s) have negative battery_needed "
          f"(likely sensor noise), flight ids: {bad}")
 
flight_summary.to_parquet(OUT_PUT, index=False, compression= 'zstd')
print(f"Saved {flight_summary.shape[0]} flights x {flight_summary.shape[1]} cols -> {OUT_PUT}")

[warning] 3 flight(s) have negative battery_needed (likely sensor noise), flight ids: [211, 212, 213]
Saved 209 flights x 21 cols -> ./data/flight.parquet


In [71]:
df.columns

Index(['flight', 'time', 'wind_speed', 'wind_angle', 'battery_voltage',
       'battery_current', 'position_x', 'position_y', 'position_z',
       'orientation_x', 'orientation_y', 'orientation_z', 'orientation_w',
       'velocity_x', 'velocity_y', 'velocity_z', 'angular_x', 'angular_y',
       'angular_z', 'linear_acceleration_x', 'linear_acceleration_y',
       'linear_acceleration_z', 'speed', 'payload', 'altitude', 'date',
       'time_day', 'route', 'dt', 'capacity', 'wind_x', 'wind_y',
       'velocity_magnitude', 'acceleration_magnitude'],
      dtype='str')

In [73]:
selection_cols = ["position_z", "wind_speed", "wind_angle", "payload", "speed", 'wind_x', 'wind_y', 'velocity_magnitude', 'acceleration_magnitude', 'capacity']

In [74]:
X = df[selection_cols]
X

,position_z,wind_speed,wind_angle,payload,speed,wind_x,wind_y,velocity_magnitude,acceleration_magnitude,capacity
0,269.332402,0.1,12.0,0.0,4.0,0.020791,0.097815,0.017495,9.811876,0.000000
1,269.332056,0.1,3.0,0.0,4.0,0.005234,0.099863,0.018517,9.811084,0.000005
2,269.333081,0.1,352.0,0.0,4.0,-0.013917,0.099027,0.019381,9.810283,0.000003
3,269.334648,0.1,354.0,0.0,4.0,-0.010453,0.099452,0.017773,9.810998,0.000005
4,269.336178,0.1,359.0,0.0,4.0,-0.001745,0.099985,0.018388,9.812861,0.000002
...,...,...,...,...,...,...,...,...,...,...
257891,271.560190,1.1,198.0,0.0,10.0,-0.339919,-1.046162,0.031708,9.810635,0.000003
257892,271.571983,1.1,196.0,0.0,10.0,-0.303201,-1.057388,0.029964,9.807152,0.000003
257893,271.584533,1.2,189.0,0.0,10.0,-0.187721,-1.185226,0.029260,9.809906,0.000006
257894,271.588050,1.1,187.0,0.0,10.0,-0.134056,-1.091801,0.034875,9.807902,0.000007
